# Lesson 19: Securing AI Agents with Cryptographic Receipts

In this notebook, you will:
1. Produce **Ed25519 (`PyNaCl`) + RFC 8785 (`jcs`)** signed decision receipts (`agent.tool_call.v1`).
2. Detect byte tampering and attacker re-signing (`04_resigned_receipt`), verify a 3-receipt hash chain, and enforce dual-registry `human.approval.v1` action-digest binding.

In [ ]:
!pip install -q -U pynacl jcs

In [ ]:
import base64, hashlib
from nacl import signing
from jcs import canonicalize

def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def sha256_canonical(obj) -> str:
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

signing_key = signing.SigningKey.generate()
pinned_public_key = b64url_nopad(bytes(signing_key.verify_key))

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical({"origin": "SYD", "destination": "LAX"}),
    "sequence": 0,
    "previous_receipt_hash": None,
}
sig = signing_key.sign(canonicalize(payload)).signature
receipt = {**payload, "signature": {"alg": "EdDSA", "sig": b64url_nopad(sig), "public_key": pinned_public_key}}

# Verify against pinned public key
vk = signing.VerifyKey(base64.urlsafe_b64decode(pinned_public_key + "=="[:(4 - len(pinned_public_key) % 4) % 4]))
vk.verify(canonicalize(payload), signing_key.sign(canonicalize(payload)).signature)
print("Ed25519 + RFC 8785 JCS Receipt Verified Offline!")